# Cross-Standard Mapping
Map data between OMOP CDM, FHIR R4, HL7 v2.5.1, and OpenEHR using Portiere's CrossStandardMapper.

In [1]:
import sys
sys.path.insert(0, "../../packages/sdk/src")

from portiere.local.cross_mapper import CrossStandardMapper, list_crossmaps
from portiere.local.transforms import TransformRegistry
from portiere.models.target_model import get_target_model

## Available Crossmaps
Portiere ships with built-in crossmap definitions for common standard pairs.

In [2]:
for cm in list_crossmaps():
    print(f"  {cm['source']} → {cm['target']}  ({cm['file']})")

  fhir_r4 → omop_cdm_v5.4  (fhir_r4_to_omop.yaml)
  fhir_r4 → openehr_1.0.4  (fhir_r4_to_openehr.yaml)
  hl7v2_2.5.1 → fhir_r4  (hl7v2_to_fhir_r4.yaml)
  omop_cdm_v5.4 → fhir_r4  (omop_to_fhir_r4.yaml)
  omop_cdm_v5.4 → openehr_1.0.4  (omop_to_openehr.yaml)


## OMOP CDM → FHIR R4
Map OMOP person records to FHIR Patient resources.

In [3]:
mapper = CrossStandardMapper("omop_cdm_v5.4", "fhir_r4")

# Map a person record
person = {
    "person_id": 12345,
    "gender_concept_id": 8507,
    "birth_datetime": "1990-05-15",
    "person_source_value": "PAT-001",
}
patient = mapper.map_record("person", person)
print("OMOP person →", patient)

# Entity mapping overview
print("\nEntity map:")
for src, tgt in mapper.get_entity_map().items():
    print(f"  {src} → {tgt}")

OMOP person → {'id': '12345', 'gender': 'male', 'birthDate': '1990-05-15', 'identifier': 'PAT-001', 'extension': None}

Entity map:
  person → Patient
  visit_occurrence → Encounter
  condition_occurrence → Condition
  drug_exposure → MedicationRequest
  measurement → Observation
  procedure_occurrence → Procedure
  observation → Observation


## HL7 v2 → FHIR R4
Map HL7 v2 PID segments to FHIR Patient resources.

In [4]:
mapper_v2 = CrossStandardMapper("hl7v2_2.5.1", "fhir_r4")

pid_segment = {
    "patient_id": "MRN-9999",
    "patient_name": "DOE^JOHN",
    "date_of_birth": "19900515",
    "administrative_sex": "M",
    "phone_home": "555-0100",
    "marital_status": "S",
}
fhir_patient = mapper_v2.map_record("PID", pid_segment)
print("HL7 v2 PID →", fhir_patient)

HL7 v2 PID → {'identifier': '', 'name': 'DOE^JOHN', 'birthDate': '19900515', 'gender': 'male', 'extension': None, 'address': '', 'telecom': '555-0100', 'maritalStatus': {'coding': [{'system': 'http://terminology.hl7.org/CodeSystem/v3-MaritalStatus', 'code': 'S', 'display': ''}], 'text': 'S'}, 'deceasedDateTime': None, 'deceasedBoolean': None}


## OMOP CDM → OpenEHR
Map OMOP measurement records to OpenEHR laboratory test results.

In [5]:
mapper_oehr = CrossStandardMapper("omop_cdm_v5.4", "openehr_1.0.4")

measurement = {
    "measurement_source_value": "2160-0",
    "measurement_date": "2024-01-15",
    "value_as_number": 1.2,
    "unit_source_value": "mg/dL",
    "range_low": 0.7,
    "range_high": 1.3,
}
lab_result = mapper_oehr.map_record("measurement", measurement)
print("OMOP measurement →")
import json
print(json.dumps(lab_result, indent=2))

OMOP measurement →
{
  "laboratory_test_result": {
    "test_name": {
      "_type": "DV_CODED_TEXT",
      "value": "2160-0",
      "defining_code": {
        "terminology_id": {
          "value": "LOINC"
        },
        "code_string": "2160-0"
      }
    },
    "test_timestamp": "2024-01-15",
    "result_value": {
      "_type": "DV_QUANTITY",
      "magnitude": 1.2,
      "units": "mg/dL"
    },
    "result_unit": "mg/dL",
    "reference_range_lower": 0.7,
    "reference_range_upper": 1.3
  }
}


## FHIR R4 → OMOP CDM
Reverse mapping from FHIR back to OMOP.

In [6]:
mapper_rev = CrossStandardMapper("fhir_r4", "omop_cdm_v5.4")

fhir_patient = {
    "id": "42",
    "gender": "female",
    "birthDate": "1985-03-22",
    "identifier": "PAT-002",
}
omop_person = mapper_rev.map_record("Patient", fhir_patient)
print("FHIR Patient →", omop_person)

FHIR Patient → {'person_id': 42, 'gender_concept_id': 8532, 'birth_datetime': '1985-03-22', 'person_source_value': 'PAT-002'}


## Batch Mapping
Map multiple records at once.

In [7]:
persons = [
    {"person_id": 1, "gender_concept_id": 8507, "birth_datetime": "1980-01-01"},
    {"person_id": 2, "gender_concept_id": 8532, "birth_datetime": "1992-06-15"},
    {"person_id": 3, "gender_concept_id": 8551, "birth_datetime": "2000-12-30"},
]

mapper = CrossStandardMapper("omop_cdm_v5.4", "fhir_r4")
patients = mapper.map_records("person", persons)
for p in patients:
    print(p)

{'id': '1', 'gender': 'male', 'birthDate': '1980-01-01', 'identifier': '', 'extension': None}
{'id': '2', 'gender': 'female', 'birthDate': '1992-06-15', 'identifier': '', 'extension': None}
{'id': '3', 'gender': 'other', 'birthDate': '2000-12-30', 'identifier': '', 'extension': None}


## Mapping Report
Inspect mapping coverage between two standards.

In [8]:
report = mapper.get_mapping_report()
print(f"Source: {report['source_standard']}")
print(f"Target: {report['target_standard']}")
print(f"Entity mappings: {len(report['entity_mappings'])}")
print(f"Field mappings: {report['field_mappings']}")
print(f"Unmapped source fields: {len(report['unmapped_source_fields'])}")
print(f"Unmapped target fields: {len(report['unmapped_target_fields'])}")

Source: omop_cdm_v5.4
Target: fhir_r4
Entity mappings: 7
Field mappings: 36
Unmapped source fields: 33
Unmapped target fields: 35


## Built-in Transforms
List all available transform functions.

In [9]:
tr = TransformRegistry()
print("Available transforms:")
for name in tr.list_transforms():
    print(f"  - {name}")

Available transforms:
  - bool
  - codeable_concept
  - dv_coded_text
  - dv_quantity
  - fhir_date
  - fhir_period
  - fhir_reference
  - float
  - format
  - hl7v2_field
  - int
  - passthrough
  - str
  - value_map
  - vocabulary_lookup


---

## Engine-Integrated Pipeline: Cross-Standard Mapping in Data Pipelines

The examples above show standalone cross-mapping on raw dicts. In practice, you'll cross-map **within a data pipeline** — reading data with an engine, running ETL to produce a standard output, then cross-mapping that output to another standard.

Portiere's `cross_map()` works natively with **Polars**, **Pandas**, and **Spark** DataFrames. The return type matches the input type, so your pipeline stays consistent with your engine of choice.

### Pipeline with Polars Engine (Default)

Initialize a **cross-map project** with `task="cross_map"` — this declares the project's intent upfront.
The `source_standard` and `target_model` are set once at init, so `project.cross_map()` calls don't
need to repeat them.

In [10]:
import portiere
from portiere.engines import PolarsEngine
from portiere.config import PortiereConfig, KnowledgeLayerConfig
from portiere.local.cross_mapper import CrossStandardMapper

# --- Step 1: Run ETL pipeline to produce OMOP output ---
config = PortiereConfig(
    target_model="omop_cdm_v5.4",
    knowledge_layer=KnowledgeLayerConfig(backend="bm25s"),
)
project_etl = portiere.init(
    name="Hospital OMOP ETL",
    task="standardize",
    engine=PolarsEngine(),
    target_model="omop_cdm_v5.4",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    config=config,
)

patients = project_etl.add_source("example_assets/13_cross_standard_mapping/patients.csv", name="patients")
schema_map = project_etl.map_schema(patients)
concept_map = project_etl.map_concepts(source=patients)
etl_result = project_etl.run_etl(
    patients,
    output_dir="./omop_output",
    schema_mapping=schema_map,
    concept_mapping=concept_map,
)
print(f"ETL complete: {len(etl_result.tables)} tables written with {project_etl.engine.engine_name} engine")

# --- Step 2: Cross-map OMOP output → FHIR R4 using a cross_map project ---
import polars as pl

# Initialize a cross-map project — source/target set once
project_xmap = portiere.init(
    name="OMOP to FHIR Cross-Map",
    task="cross_map",
    source_standard="omop_cdm_v5.4",
    target_model="fhir_r4",
    engine=PolarsEngine(),
    config=config,
)

# Read the OMOP person table produced by ETL
omop_persons = pl.read_csv("./omop_output/person.csv")
print(f"OMOP persons: {omop_persons.shape[0]} rows (Polars DataFrame)")

# Cross-map — no need to repeat source/target, project knows its task
fhir_patients = project_xmap.cross_map(
    source_entity="person",
    data=omop_persons,
)
print(f"FHIR patients: {fhir_patients.shape[0]} rows (type: {type(fhir_patients).__name__})")
print(fhir_patients.head(5))

2026-04-16 00:34:21 [info     ] PolarsEngine initialized      


2026-04-16 00:34:21 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


2026-04-16 00:34:21 [info     ] project.source_added           project='Hospital OMOP ETL' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:34:24 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


...

2026-04-16 00:34:26 [info     ] project.etl_complete           output_dir=./omop_output project='Hospital OMOP ETL'


ETL complete: 2 tables written with polars engine
2026-04-16 00:34:26 [info     ] PolarsEngine initialized      


2026-04-16 00:34:26 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


OMOP persons: 15 rows (Polars DataFrame)
2026-04-16 00:34:26 [info     ] local_storage.cross_mapping_saved project='OMOP to FHIR Cross-Map' runs_count=2


FHIR patients: 15 rows (type: DataFrame)
shape: (5, 5)
┌──────┬─────────┬────────────┬────────────┬───────────────────────────┐
│ id   ┆ gender  ┆ birthDate  ┆ identifier ┆ extension                 │
│ ---  ┆ ---     ┆ ---        ┆ ---        ┆ ---                       │
│ str  ┆ str     ┆ str        ┆ str        ┆ str                       │
╞══════╪═════════╪════════════╪════════════╪═══════════════════════════╡
│ P001 ┆ unknown ┆ 1958-03-14 ┆ James      ┆ White                     │
│ P002 ┆ unknown ┆ 1972-08-22 ┆ Maria      ┆ White                     │
│ P003 ┆ unknown ┆ 1965-11-30 ┆ Robert     ┆ Black or African American │
│ P004 ┆ unknown ┆ 1980-05-17 ┆ Linda      ┆ Asian                     │
│ P005 ┆ unknown ┆ 1948-01-09 ┆ William    ┆ White                     │
└──────┴─────────┴────────────┴────────────┴───────────────────────────┘


### Cross-Map Multiple OMOP Tables in a Polars Pipeline

After ETL, cross-map all relevant OMOP tables to FHIR R4 in one pass.

In [11]:
import polars as pl
from pathlib import Path

mapper = CrossStandardMapper("omop_cdm_v5.4", "fhir_r4")

# OMOP → FHIR entity mapping
entity_map = mapper.get_entity_map()
print("Entity mapping:")
for omop_table, fhir_resource in entity_map.items():
    print(f"  {omop_table} → {fhir_resource}")

# Cross-map each OMOP table that was produced by ETL
omop_dir = Path("./omop_output")
fhir_output = Path("./fhir_output")
fhir_output.mkdir(exist_ok=True)

for omop_table, fhir_resource in entity_map.items():
    omop_file = omop_dir / f"{omop_table}.csv"
    if not omop_file.exists():
        continue

    # Read OMOP table with Polars
    df = pl.read_csv(omop_file)
    print(f"\n{omop_table} ({df.shape[0]} rows) → {fhir_resource}")

    # Cross-map — input Polars → output Polars
    fhir_df = mapper.map_dataframe(omop_table, df)
    print(f"  Output: {fhir_df.shape[0]} rows, columns: {fhir_df.columns}")

    # Write FHIR output as Parquet (stays in Polars pipeline)
    fhir_df.write_parquet(fhir_output / f"{fhir_resource}.parquet")

print(f"\nFHIR resources written to {fhir_output}/")

Entity mapping:
  person → Patient
  visit_occurrence → Encounter
  condition_occurrence → Condition
  drug_exposure → MedicationRequest
  measurement → Observation
  procedure_occurrence → Procedure
  observation → Observation

person (15 rows) → Patient
  Output: 15 rows, columns: ['id', 'gender', 'birthDate', 'identifier', 'extension']

visit_occurrence (20 rows) → Encounter
  Output: 20 rows, columns: ['id', 'subject', 'type', 'period', 'identifier']


InvalidOperationError: Unable to write struct type with no child field to Parquet. Consider adding a dummy child field.

### Pipeline with Spark Engine (Distributed)

Same pipeline using SparkEngine — ideal for large-scale hospital datasets. Cross-map accepts Spark DataFrames and returns Spark DataFrames.

In [12]:
try:
    from pyspark.sql import SparkSession
    from portiere.engines import SparkEngine

    spark = SparkSession.builder \
        .appName("portiere-cross-map") \
        .master("local[*]") \
        .getOrCreate()

    project_spark = portiere.init(
        name="Hospital Spark Pipeline",
        task="standardize",
        engine=SparkEngine(spark),
        target_model="omop_cdm_v5.4",
    )
    print(f"Spark project: {project_spark.name}")
    SPARK_AVAILABLE = True
except ImportError:
    SPARK_AVAILABLE = False
    spark = None
    project_spark = None
    print("PySpark not installed — Spark cells will be skipped. Install with: pip install pyspark")


PySpark not installed — Spark cells will be skipped. Install with: pip install pyspark


### Pipeline with Pandas Engine (Lightweight)

For small datasets or quick prototyping with Pandas.

In [13]:
import pandas as pd
from portiere.engines import PandasEngine

# --- Step 1: Initialize project with Pandas engine for ETL ---
project_pd = portiere.init(
    name="Quick Pandas Pipeline",
    task="standardize",
    engine=PandasEngine(),
    target_model="omop_cdm_v5.4",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    config=config,
)

# --- Step 2: Run pipeline to OMOP with Pandas engine ---
patients_src = project_pd.add_source("example_assets/13_cross_standard_mapping/patients.csv", name="patients")
schema_map = project_pd.map_schema(patients_src)
concept_map = project_pd.map_concepts(source=patients_src)
etl_result = project_pd.run_etl(
    patients_src,
    output_dir="./pandas_omop_output",
    schema_mapping=schema_map,
    concept_mapping=concept_map,
)
print(f"Pandas ETL complete: {len(etl_result.tables)} tables")

# --- Step 3: Cross-map OMOP → FHIR R4 using a cross_map project ---
project_pd_xmap = portiere.init(
    name="Pandas OMOP→FHIR Cross-Map",
    task="cross_map",
    source_standard="omop_cdm_v5.4",
    target_model="fhir_r4",
    engine=PandasEngine(),
    config=config,
)

omop_persons_pd = pd.read_csv("./pandas_omop_output/person.csv")
print(f"OMOP persons (Pandas): {len(omop_persons_pd)} rows")

# Cross-map — project knows source/target
fhir_patients_pd = project_pd_xmap.cross_map(
    source_entity="person",
    data=omop_persons_pd,
)
print(f"FHIR patients (Pandas): {len(fhir_patients_pd)} rows")
print(f"Type: {type(fhir_patients_pd).__name__}")
fhir_patients_pd.head()

2026-04-16 00:34:27 [info     ] PandasEngine initialized      


2026-04-16 00:34:27 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


2026-04-16 00:34:27 [info     ] project.source_added           project='Quick Pandas Pipeline' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:34:27 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


...

2026-04-16 00:34:27 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


OMOP persons (Pandas): 15 rows


2026-04-16 00:34:27 [info     ] local_storage.cross_mapping_saved project='Pandas OMOP→FHIR Cross-Map' runs_count=2


FHIR patients (Pandas): 15 rows
Type: DataFrame


,id,gender,birthDate,identifier,extension
0,P001,unknown,1958-03-14,James,White
1,P002,unknown,1972-08-22,Maria,White
2,P003,unknown,1965-11-30,Robert,Black or African American
3,P004,unknown,1980-05-17,Linda,Asian
4,P005,unknown,1948-01-09,William,White


### Multi-Hop Cross-Mapping: OMOP → FHIR → OpenEHR

Chain cross-maps to go from OMOP through FHIR to OpenEHR. Each step uses the same engine-native DataFrame type.

In [14]:
import polars as pl

# Simulate OMOP condition_occurrence data
omop_conditions = pl.DataFrame({
    "condition_occurrence_id": [1, 2, 3],
    "person_id": [101, 102, 103],
    "condition_concept_id": [201826, 312327, 4329847],
    "condition_source_value": ["E11.9", "I10", "J44.1"],
    "condition_start_date": ["2024-01-15", "2024-02-20", "2024-03-10"],
    "condition_type_concept_id": [32817, 32817, 32817],
})
print("Step 0 — OMOP condition_occurrence (Polars):")
print(omop_conditions)

# Step 1: OMOP → FHIR R4
mapper_omop_fhir = CrossStandardMapper("omop_cdm_v5.4", "fhir_r4")
fhir_conditions = mapper_omop_fhir.map_dataframe("condition_occurrence", omop_conditions)
print("\nStep 1 — FHIR Condition (Polars):")
print(fhir_conditions)

# Step 2: FHIR R4 → OpenEHR
mapper_fhir_oehr = CrossStandardMapper("fhir_r4", "openehr_1.0.4")
openehr_diagnoses = mapper_fhir_oehr.map_dataframe("Condition", fhir_conditions)
print("\nStep 2 — OpenEHR problem_diagnosis (Polars):")
print(openehr_diagnoses)
print(f"\nAll DataFrames stay as Polars: {type(openehr_diagnoses).__name__}")

Step 0 — OMOP condition_occurrence (Polars):
shape: (3, 6)
┌─────────────────┬───────────┬─────────────────┬────────────────┬────────────────┬────────────────┐
│ condition_occur ┆ person_id ┆ condition_conce ┆ condition_sour ┆ condition_star ┆ condition_type │
│ rence_id        ┆ ---       ┆ pt_id           ┆ ce_value       ┆ t_date         ┆ _concept_id    │
│ ---             ┆ i64       ┆ ---             ┆ ---            ┆ ---            ┆ ---            │
│ i64             ┆           ┆ i64             ┆ str            ┆ str            ┆ i64            │
╞═════════════════╪═══════════╪═════════════════╪════════════════╪════════════════╪════════════════╡
│ 1               ┆ 101       ┆ 201826          ┆ E11.9          ┆ 2024-01-15     ┆ 32817          │
│ 2               ┆ 102       ┆ 312327          ┆ I10            ┆ 2024-02-20     ┆ 32817          │
│ 3               ┆ 103       ┆ 4329847         ┆ J44.1          ┆ 2024-03-10     ┆ 32817          │
└─────────────────┴───────────┴─


Step 1 — FHIR Condition (Polars):
shape: (3, 5)
┌─────┬─────────────────┬─────────────────────────────────┬───────────────┬────────────┐
│ id  ┆ subject         ┆ code                            ┆ onsetDateTime ┆ identifier │
│ --- ┆ ---             ┆ ---                             ┆ ---           ┆ ---        │
│ str ┆ struct[1]       ┆ struct[2]                       ┆ str           ┆ str        │
╞═════╪═════════════════╪═════════════════════════════════╪═══════════════╪════════════╡
│ 1   ┆ {"Patient/101"} ┆ {[{"http://snomed.info/sct","2… ┆ 2024-01-15    ┆ E11.9      │
│ 2   ┆ {"Patient/102"} ┆ {[{"http://snomed.info/sct","3… ┆ 2024-02-20    ┆ I10        │
│ 3   ┆ {"Patient/103"} ┆ {[{"http://snomed.info/sct","4… ┆ 2024-03-10    ┆ J44.1      │
└─────┴─────────────────┴─────────────────────────────────┴───────────────┴────────────┘



Step 2 — OpenEHR problem_diagnosis (Polars):
shape: (3, 1)
┌─────────────────────────────────┐
│ problem_diagnosis               │
│ ---                             │
│ struct[4]                       │
╞═════════════════════════════════╡
│ {{"DV_CODED_TEXT","{'coding': … │
│ {{"DV_CODED_TEXT","{'coding': … │
│ {{"DV_CODED_TEXT","{'coding': … │
└─────────────────────────────────┘

All DataFrames stay as Polars: DataFrame


### Cross-Map with VocabularyBridge in Pipeline

Use VocabularyBridge alongside cross-mapping to translate concept codes between vocabularies during the cross-map step.

In [15]:
from portiere.knowledge.vocabulary_bridge import VocabularyBridge
import polars as pl

# Initialize VocabularyBridge for concept translation
bridge = VocabularyBridge(
    athena_path="./example_assets/vocabulary_download_v5/",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
)

# Create cross-mapper with vocabulary bridge
mapper_with_vocab = CrossStandardMapper(
    "omop_cdm_v5.4", "fhir_r4",
    vocabulary_bridge=bridge,
)

# OMOP measurement data (Polars)
omop_measurements = pl.DataFrame({
    "measurement_id": [1, 2, 3],
    "person_id": [101, 102, 103],
    "measurement_concept_id": [3004249, 3006322, 3013650],
    "measurement_source_value": ["2160-0", "4548-4", "789-8"],
    "value_as_number": [1.2, 6.5, 5.1],
    "unit_source_value": ["mg/dL", "%", "10*3/uL"],
    "measurement_date": ["2024-01-15", "2024-02-01", "2024-03-05"],
})

# Cross-map with vocabulary lookup — OMOP concept_ids resolved to SNOMED/LOINC codes
fhir_observations = mapper_with_vocab.map_dataframe("measurement", omop_measurements)
print("FHIR Observations (with vocabulary-resolved codes):")
print(fhir_observations)

# VocabularyBridge can also produce FHIR CodeableConcept or OpenEHR DV_CODED_TEXT
cc = bridge.concept_to_codeable_concept(3004249)
print(f"\nFHIR CodeableConcept for creatinine: {cc}")

dv = bridge.concept_to_dv_coded_text(3004249)
print(f"OpenEHR DV_CODED_TEXT for creatinine: {dv}")

FHIR Observations (with vocabulary-resolved codes):
shape: (3, 6)
┌─────┬─────────────────┬─────────────────────────┬───────────────────┬───────────────┬────────────┐
│ id  ┆ subject         ┆ code                    ┆ effectiveDateTime ┆ valueQuantity ┆ identifier │
│ --- ┆ ---             ┆ ---                     ┆ ---               ┆ ---           ┆ ---        │
│ str ┆ struct[1]       ┆ struct[2]               ┆ str               ┆ f64           ┆ str        │
╞═════╪═════════════════╪═════════════════════════╪═══════════════════╪═══════════════╪════════════╡
│ 1   ┆ {"Patient/101"} ┆ {[{"http://loinc.org"," ┆ 2024-01-15        ┆ 1.2           ┆ 2160-0     │
│     ┆                 ┆ 3004249…                ┆                   ┆               ┆            │
│ 2   ┆ {"Patient/102"} ┆ {[{"http://loinc.org"," ┆ 2024-02-01        ┆ 6.5           ┆ 4548-4     │
│     ┆                 ┆ 3006322…                ┆                   ┆               ┆            │
│ 3   ┆ {"Patient/103"} ┆

2026-04-16 00:34:40 [info     ] vocabulary_bridge.concepts_loaded count=1784037


2026-04-16 00:34:40 [info     ] vocabulary_bridge.loading_relationships path=example_assets/vocabulary_download_v5/CONCEPT_RELATIONSHIP.csv


2026-04-16 00:35:36 [info     ] vocabulary_bridge.relationships_loaded count=3635306



FHIR CodeableConcept for creatinine: {'coding': [{'system': 'http://loinc.org', 'code': '8480-6', 'display': 'Systolic blood pressure'}], 'text': 'Systolic blood pressure'}
OpenEHR DV_CODED_TEXT for creatinine: {'_type': 'DV_CODED_TEXT', 'value': 'Systolic blood pressure', 'defining_code': {'terminology_id': {'value': 'LOINC'}, 'code_string': '8480-6'}}


### Engine Comparison: Same Cross-Map, Different Engines

Quick reference showing the same cross-mapping operation with each engine type.

In [16]:
import polars as pl
import pandas as pd

mapper = CrossStandardMapper("omop_cdm_v5.4", "fhir_r4")

sample_data = [
    {"person_id": 1, "gender_concept_id": 8507, "birth_datetime": "1980-01-01"},
    {"person_id": 2, "gender_concept_id": 8532, "birth_datetime": "1992-06-15"},
]

# --- Polars ---
polars_df = pl.DataFrame(sample_data)
polars_result = mapper.map_dataframe("person", polars_df)
print(f"Polars input  → Polars output:  {type(polars_result).__name__}")
print(polars_result)

# --- Pandas ---
pandas_df = pd.DataFrame(sample_data)
pandas_result = mapper.map_dataframe("person", pandas_df)
print(f"\nPandas input  → Pandas output:  {type(pandas_result).__name__}")
print(pandas_result)

# --- Spark ---
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("demo").master("local[*]").getOrCreate()
# spark_df = spark.createDataFrame(sample_data)
# spark_result = mapper.map_dataframe("person", spark_df)
# print(f"\nSpark input   → Spark output:   {type(spark_result).__name__}")
# spark_result.show()
# spark.stop()

Polars input  → Polars output:  DataFrame
shape: (2, 5)
┌─────┬────────┬────────────┬────────────┬───────────┐
│ id  ┆ gender ┆ birthDate  ┆ identifier ┆ extension │
│ --- ┆ ---    ┆ ---        ┆ ---        ┆ ---       │
│ str ┆ str    ┆ str        ┆ str        ┆ null      │
╞═════╪════════╪════════════╪════════════╪═══════════╡
│ 1   ┆ male   ┆ 1980-01-01 ┆            ┆ null      │
│ 2   ┆ female ┆ 1992-06-15 ┆            ┆ null      │
└─────┴────────┴────────────┴────────────┴───────────┘

Pandas input  → Pandas output:  DataFrame
  id  gender   birthDate identifier extension
0  1    male  1980-01-01                 None
1  2  female  1992-06-15                 None


---

## Server-Side Tracking: Cross-Mapping Audit Trail

Every `project.cross_map()` call is automatically persisted — locally as YAML or to the cloud API. This is consistent with how schema and concept mappings are tracked.

### Local Mode
Runs are saved to `{project_dir}/cross_mappings/cross_mapping.yaml`.

### Cloud Mode
Runs are synced to the Portiere API via:
- `POST /api/v1/cross-mapping/projects/{id}/map` — map and persist
- `GET /api/v1/cross-mapping/projects/{id}` — view run history
- `POST /api/v1/sync/projects/{id}/cross-mappings/bulk` — bulk sync

In [17]:
import portiere
from portiere.engines import PolarsEngine
from portiere.config import PortiereConfig, KnowledgeLayerConfig
from portiere.models.cross_mapping import CrossMapping, CrossMappingRun
import polars as pl

# --- Initialize a cross-map project — runs are automatically tracked ---
config = PortiereConfig(
    target_model="fhir_r4",
    knowledge_layer=KnowledgeLayerConfig(backend="bm25s"),
)
project = portiere.init(
    name="Tracked Pipeline",
    task="cross_map",
    source_standard="omop_cdm_v5.4",
    target_model="fhir_r4",
    engine=PolarsEngine(),
    config=config,
)

# Simulate OMOP person data
omop_persons = pl.DataFrame({
    "person_id": [1, 2, 3],
    "gender_concept_id": [8507, 8532, 8507],
    "birth_datetime": ["1980-01-01", "1992-06-15", "2000-12-30"],
})

# Every cross_map() call is persisted to storage — no need to specify source/target
fhir_patients = project.cross_map(
    source_entity="person",
    data=omop_persons,
)
print("Cross-map result:")
print(fhir_patients)

# Do another cross-map for conditions
omop_conditions = pl.DataFrame({
    "condition_occurrence_id": [10, 20],
    "person_id": [1, 2],
    "condition_concept_id": [201826, 312327],
    "condition_source_value": ["E11.9", "I10"],
    "condition_start_date": ["2024-01-15", "2024-02-20"],
    "condition_type_concept_id": [32817, 32817],
})
fhir_conditions = project.cross_map(
    source_entity="condition_occurrence",
    data=omop_conditions,
)
print("\nCondition cross-map result:")
print(fhir_conditions)

2026-04-16 00:35:36 [info     ] PolarsEngine initialized      


2026-04-16 00:35:36 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


2026-04-16 00:35:36 [info     ] local_storage.cross_mapping_saved project='Tracked Pipeline' runs_count=5


Cross-map result:
shape: (3, 5)
┌─────┬────────┬────────────┬────────────┬───────────┐
│ id  ┆ gender ┆ birthDate  ┆ identifier ┆ extension │
│ --- ┆ ---    ┆ ---        ┆ ---        ┆ ---       │
│ str ┆ str    ┆ str        ┆ str        ┆ null      │
╞═════╪════════╪════════════╪════════════╪═══════════╡
│ 1   ┆ male   ┆ 1980-01-01 ┆            ┆ null      │
│ 2   ┆ female ┆ 1992-06-15 ┆            ┆ null      │
│ 3   ┆ male   ┆ 2000-12-30 ┆            ┆ null      │
└─────┴────────┴────────────┴────────────┴───────────┘


2026-04-16 00:35:36 [info     ] local_storage.cross_mapping_saved project='Tracked Pipeline' runs_count=6



Condition cross-map result:
shape: (2, 5)
┌─────┬───────────────┬─────────────────────────────────┬───────────────┬────────────┐
│ id  ┆ subject       ┆ code                            ┆ onsetDateTime ┆ identifier │
│ --- ┆ ---           ┆ ---                             ┆ ---           ┆ ---        │
│ str ┆ struct[1]     ┆ struct[2]                       ┆ str           ┆ str        │
╞═════╪═══════════════╪═════════════════════════════════╪═══════════════╪════════════╡
│ 10  ┆ {"Patient/1"} ┆ {[{"http://snomed.info/sct","2… ┆ 2024-01-15    ┆ E11.9      │
│ 20  ┆ {"Patient/2"} ┆ {[{"http://snomed.info/sct","3… ┆ 2024-02-20    ┆ I10        │
└─────┴───────────────┴─────────────────────────────────┴───────────────┴────────────┘


### Inspecting the Cross-Mapping History

After running cross-maps, load the history from storage to see what was tracked.

In [18]:
# Load cross-mapping history from storage
history = project._storage.load_cross_mapping(project.name)

print(f"Cross-mapping runs: {len(history.runs)}")
print()
for i, run in enumerate(history.runs, 1):
    print(f"Run {i}:")
    print(f"  {run.source_standard} → {run.target_standard}")
    print(f"  Entity: {run.source_entity} → {run.target_entity}")
    print(f"  Records: {run.record_count}")
    print(f"  Status: {run.status}")

# Summary
print("\nSummary:")
summary = history.summary()
print(f"  Total runs: {summary['total_runs']}")
print(f"  Total records mapped: {summary['total_records']}")
print(f"  Standard pairs: {summary['standard_pairs']}")

Cross-mapping runs: 6

Run 1:
  omop_cdm_v5.4 → fhir_r4
  Entity: person → Patient
  Records: 3
  Status: completed
Run 2:
  omop_cdm_v5.4 → fhir_r4
  Entity: condition_occurrence → Condition
  Records: 2
  Status: completed
Run 3:
  omop_cdm_v5.4 → fhir_r4
  Entity: person → Patient
  Records: 3
  Status: completed
Run 4:
  omop_cdm_v5.4 → fhir_r4
  Entity: condition_occurrence → Condition
  Records: 2
  Status: completed
Run 5:
  omop_cdm_v5.4 → fhir_r4
  Entity: person → Patient
  Records: 3
  Status: completed
Run 6:
  omop_cdm_v5.4 → fhir_r4
  Entity: condition_occurrence → Condition
  Records: 2
  Status: completed

Summary:
  Total runs: 6
  Total records mapped: 15
  Standard pairs: ['omop_cdm_v5.4 → fhir_r4']


### Cloud Mode: Tracked Cross-Mapping via API

When using a cloud backend, cross-mapping runs are synced to the Portiere API. You can also use the API directly for tracked cross-mapping.

In [19]:
# Cross-standard mapping via REST API requires the Portiere server running locally.
# This cell is illustrative — it shows the API contract but is skipped in notebook execution.
print("Skipped: cross-map REST API requires running Portiere server (portiere serve)")
# Example request (run when server is available):
# import requests
# response = requests.post('http://localhost:8000/api/v1/cross-mapping/...', json={...})


Skipped: cross-map REST API requires running Portiere server (portiere serve)


### Cloud SDK: Automatic Sync

When using the Portiere Client, `project.cross_map()` automatically syncs runs to the cloud — no manual API calls needed.

In [20]:
# Cloud-sync demo (Portiere Cloud only — skipped in open-source SDK)
try:
    from portiere.client import Client
    import polars as pl

    client = Client(api_key="pt_sk_live_...")
    print("Cloud client initialized (demo — requires real API key)")
except NotImplementedError as e:
    print(f"Cloud features not available in open-source SDK: {e}")


Cloud features not available in open-source SDK: Cloud features are not available in the open-source SDK. For cloud storage, sync, and managed inference, see https://portiere.io
